In [1]:
pip install osmnx folium networkx numpy


In [3]:
import math
import random
import numpy as np
import networkx as nx
import osmnx as ox
import folium
from folium.features import DivIcon

random.seed(42)
np.random.seed(42)

# ==========================================================
# CONFIG: Farmgate -> Shahbagh (change coords if needed)
# ==========================================================
START_LATLON = (23.759954868041955, 90.38945528575461)   # Farmgate
GOAL_LATLON  = (23.7385, 90.3958)                        # Shahbagh (approx)

GRAPH_RADIUS_M = 3500   # good for Farmgate->Shahbagh; increase if no route

# ==========================================================
# STEP 1: Download OSM road graph
# ==========================================================
def build_osm_graph(start_latlon, goal_latlon, radius_m=3500):
    mid_lat = (start_latlon[0] + goal_latlon[0]) / 2.0
    mid_lon = (start_latlon[1] + goal_latlon[1]) / 2.0

    # simplify=True already simplifies (don’t call simplify_graph again)
    G = ox.graph_from_point((mid_lat, mid_lon), dist=radius_m, network_type="drive", simplify=True)
    return G

G = build_osm_graph(START_LATLON, GOAL_LATLON, GRAPH_RADIUS_M)

start_node = ox.distance.nearest_nodes(G, X=START_LATLON[1], Y=START_LATLON[0])
goal_node  = ox.distance.nearest_nodes(G, X=GOAL_LATLON[1],  Y=GOAL_LATLON[0])

print("Graph nodes:", len(G.nodes), "edges:", len(G.edges))
print("Start node:", start_node, "Goal node:", goal_node)

# ==========================================================
# STEP 2: Dynamic traffic simulation -> edge cost
# ==========================================================
ROADTYPE_BASE = {
    "motorway": 0.3, "trunk": 0.35, "primary": 0.45, "secondary": 0.6,
    "tertiary": 0.75, "residential": 0.9, "service": 1.0
}

def highway_type(data):
    hw = data.get("highway", "residential")
    if isinstance(hw, list):
        hw = hw[0]
    return hw

def get_lanes(data):
    lanes = data.get("lanes", 1)
    try:
        if isinstance(lanes, list):
            lanes = lanes[0]
        lanes = int(float(lanes))
    except:
        lanes = 1
    return max(1, lanes)

def assign_dynamic_costs(G, t, severity=0.9):
    """
    cost = length * (1 + congestion)
    store:
      data["C"] = congestion coefficient
      data["cost"] = dynamic weight for routing
    """
    for u, v, k, data in G.edges(keys=True, data=True):
        length = float(data.get("length", 10.0))
        hw = highway_type(data)
        lanes = get_lanes(data)
        base = ROADTYPE_BASE.get(hw, 0.9)

        wave  = 0.5 * (1 + math.sin(2 * math.pi * (t / 6.0)))   # 0..1
        noise = random.random()                                 # 0..1

        C = base * severity * (0.6 * wave + 0.4 * noise) / lanes
        data["C"] = C
        data["cost"] = length * (1.0 + C)

assign_dynamic_costs(G, t=0)

# ==========================================================
# STEP 3: ACO shortest path
# ==========================================================
def roulette_choice(items):
    total = sum(w for _, w in items)
    if total <= 0:
        return random.choice([x for x, _ in items])
    r = random.random() * total
    cum = 0.0
    for x, w in items:
        cum += w
        if r <= cum:
            return x
    return items[-1][0]

def path_cost(G, path):
    if path is None or len(path) < 2:
        return float("inf")
    total = 0.0
    for a, b in zip(path[:-1], path[1:]):
        edata = min(G.get_edge_data(a, b).values(), key=lambda d: d["cost"])
        total += edata["cost"]
    return total

def aco_shortest_path(G, start, goal, alpha, beta, rho, num_ants=25, iters=60, Q=1.0, seed=42):
    random.seed(seed)

    tau = {}
    for u, v, k in G.edges(keys=True):
        tau[(u, v, k)] = 1.0

    best_path, best_cost = None, float("inf")

    for _ in range(iters):
        ant_paths = []

        for _ant in range(num_ants):
            current = start
            visited = {current}
            path = [current]

            for _step in range(6000):
                if current == goal:
                    break

                candidates = []
                for nxt in G.successors(current):
                    if nxt in visited:
                        continue

                    edges = G.get_edge_data(current, nxt)
                    best_k, best_d = min(edges.items(), key=lambda kv: kv[1]["cost"])
                    cost = best_d["cost"]
                    eta = 1.0 / (cost + 1e-9)

                    desir = (tau[(current, nxt, best_k)] ** alpha) * (eta ** beta)
                    candidates.append((nxt, desir))

                if not candidates:
                    break

                nxt = roulette_choice(candidates)
                path.append(nxt)
                visited.add(nxt)
                current = nxt

            if path[-1] == goal:
                c = path_cost(G, path)
                ant_paths.append((path, c))
                if c < best_cost:
                    best_cost, best_path = c, path

        # evaporate
        for key in tau:
            tau[key] *= (1.0 - rho)

        # deposit
        for pth, c in ant_paths:
            delta = Q / (c + 1e-9)
            for a, b in zip(pth[:-1], pth[1:]):
                edges = G.get_edge_data(a, b)
                best_k = min(edges.items(), key=lambda kv: kv[1]["cost"])[0]
                tau[(a, b, best_k)] += delta

    return best_path, best_cost

# ==========================================================
# STEP 4: PSO tune parameters
# ==========================================================
def clamp(x, lo, hi): return max(lo, min(hi, x))
def round_step(x, step=0.1): return math.floor(x / step + 0.5) * step

def pso_tune_params(G, start, goal,
                    swarm=12, pso_iters=12,
                    alpha_bounds=(0.5, 2.0),
                    beta_bounds=(1.0, 6.0),
                    rho_bounds=(0.1, 0.9),
                    step=0.1,
                    eval_ants=18, eval_iters=35, eval_runs=2,
                    seed=42):
    random.seed(seed)
    cache = {}

    def fitness(a, b, r):
        key = (a, b, r)
        if key in cache:
            return cache[key]
        vals = []
        for rr in range(eval_runs):
            p, c = aco_shortest_path(G, start, goal, a, b, r,
                                     num_ants=eval_ants, iters=eval_iters, seed=1000+rr)
            vals.append(c if p is not None else 1e18)
        cache[key] = float(np.mean(vals))
        return cache[key]

    particles = []
    gbest, gbest_val = None, float("inf")

    # init
    for _ in range(swarm):
        a = round_step(random.uniform(*alpha_bounds), step)
        b = round_step(random.uniform(*beta_bounds), step)
        r = round_step(random.uniform(*rho_bounds), step)
        va, vb, vr = random.uniform(-0.3,0.3), random.uniform(-0.3,0.3), random.uniform(-0.3,0.3)
        val = fitness(a,b,r)
        particles.append({"pos":[a,b,r], "vel":[va,vb,vr], "pbest":[a,b,r], "pbest_val":val})
        if val < gbest_val:
            gbest_val, gbest = val, [a,b,r]

    w, c1, c2 = 0.7, 1.5, 1.5
    for _it in range(pso_iters):
        for p in particles:
            for d in range(3):
                r1, r2 = random.random(), random.random()
                p["vel"][d] = w*p["vel"][d] + c1*r1*(p["pbest"][d]-p["pos"][d]) + c2*r2*(gbest[d]-p["pos"][d])

            p["pos"][0] = round_step(clamp(p["pos"][0] + p["vel"][0], *alpha_bounds), step)
            p["pos"][1] = round_step(clamp(p["pos"][1] + p["vel"][1], *beta_bounds), step)
            p["pos"][2] = round_step(clamp(p["pos"][2] + p["vel"][2], *rho_bounds), step)

            val = fitness(*p["pos"])
            if val < p["pbest_val"]:
                p["pbest_val"] = val
                p["pbest"] = p["pos"][:]
            if val < gbest_val:
                gbest_val, gbest = val, p["pos"][:]

    return tuple(gbest), gbest_val

# ==========================================================
# STEP 5: Baseline route (no traffic) vs ACO route (traffic)
# ==========================================================
# Baseline shortest by length only
baseline_path = nx.shortest_path(G, start_node, goal_node, weight="length")
baseline_len  = sum(min(G.get_edge_data(a,b).values(), key=lambda d: d["length"])["length"]
                    for a,b in zip(baseline_path[:-1], baseline_path[1:]))

print("Baseline length-only route:", len(baseline_path), "nodes,", f"{baseline_len:.1f}m")

# Tune ACO parameters at t=0 (traffic already assigned above)
best_params, _ = pso_tune_params(G, start_node, goal_node)
alpha, beta, rho = best_params
print("PSO tuned params:", best_params)

# Dynamic time steps and collect routes
paths = []
history = []
T = 4

for t in range(T):
    assign_dynamic_costs(G, t=t, severity=0.9)
    p, c = aco_shortest_path(G, start_node, goal_node, alpha, beta, rho,
                            num_ants=35, iters=90, seed=200+t)
    paths.append(p)
    history.append((t, c))
    print(f"t={t}: ACO cost={c:.1f} (found={p is not None})")

# ==========================================================
# STEP 6: Map Visualization (traffic edges + 2 route colors)
# ==========================================================
def node_latlon(G, n):
    return (G.nodes[n]["y"], G.nodes[n]["x"])

def edge_color_from_C(C):
    # simple threshold mapping
    # low traffic -> green, medium -> orange, high -> red
    if C < 0.25: return "green"
    if C < 0.50: return "orange"
    return "red"

m = folium.Map(location=START_LATLON, zoom_start=14, tiles="OpenStreetMap")

# markers
folium.Marker(START_LATLON, tooltip="START", icon=folium.Icon(color="green")).add_to(m)
folium.Marker(GOAL_LATLON, tooltip="GOAL", icon=folium.Icon(color="red")).add_to(m)

# --- draw traffic-colored edges (sample: draw all edges) ---
for u, v, k, data in G.edges(keys=True, data=True):
    C = float(data.get("C", 0.0))
    col = edge_color_from_C(C)
    latlon_u = node_latlon(G, u)
    latlon_v = node_latlon(G, v)
    folium.PolyLine(
        [latlon_u, latlon_v],
        color=col,
        weight=2,
        opacity=0.35
    ).add_to(m)

# --- draw baseline shortest route (green) ---
baseline_latlon = [node_latlon(G, n) for n in baseline_path]
folium.PolyLine(
    baseline_latlon,
    color="lime",
    weight=7,
    opacity=0.9,
    tooltip=f"Baseline shortest (length) ≈ {baseline_len:.0f}m"
).add_to(m)

# --- draw ACO route at final time (blue) ---
final_path = paths[-1]
if final_path is not None:
    final_latlon = [node_latlon(G, n) for n in final_path]
    final_cost = history[-1][1]
    folium.PolyLine(
        final_latlon,
        color="blue",
        weight=7,
        opacity=0.9,
        tooltip=f"ACO+PSO (traffic-aware) cost ≈ {final_cost:.0f}"
    ).add_to(m)

# --- optional: draw previous time-step ACO routes light blue ---
for t, p in enumerate(paths[:-1]):
    if p is None:
        continue
    latlon = [node_latlon(G, n) for n in p]
    folium.PolyLine(
        latlon,
        color="cadetblue",
        weight=4,
        opacity=0.45,
        tooltip=f"ACO route at t={t}"
    ).add_to(m)

# legend
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index:9999;
background: white; padding: 10px; border:2px solid gray; border-radius:8px;">
<b>Legend</b><br>
<span style="color:green;">■</span> Low traffic roads<br>
<span style="color:orange;">■</span> Medium traffic roads<br>
<span style="color:red;">■</span> High traffic roads<br><hr style="margin:6px 0;">
<span style="color:lime;">■</span> Baseline shortest (length)<br>
<span style="color:blue;">■</span> ACO+PSO best (traffic-aware)<br>
<span style="color:cadetblue;">■</span> ACO routes (earlier t)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

out = "route_compare.html"
m.save(out)
print(f"\nSaved: {out} (open in browser)")


Graph nodes: 7964 edges: 18845
Start node: 6897667505 Goal node: 3804817431
Baseline length-only route: 37 nodes, 2604.1m
PSO tuned params: (1.2000000000000002, 3.1, 0.30000000000000004)
t=0: ACO cost=inf (found=False)
t=1: ACO cost=inf (found=False)
t=2: ACO cost=5117.6 (found=True)
t=3: ACO cost=inf (found=False)

Saved: route_compare.html (open in browser)
